In [1]:
import numpy as np
import pandas as pd

from scipy.stats import norm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

In [2]:
data = pd.read_csv("../data/final_dataset.csv")

In [3]:
df = pd.DataFrame(data)
df.head()

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0


In [4]:
params = [
    "age",
    "income",
    "n_child",
    "sex",
    "type_area",
    "invalid",
    "mar_st",
    "visit_doctor",
    "work",
    "alcohol",
    "smoking",
    "phys_active",
    "is_health_very_good",
    "diploma",
]
targets = df.keys().drop(params).to_list()

In [5]:
df[params]

,age,income,n_child,sex,type_area,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_very_good,diploma
0,44.0,43000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
1,61.0,20000.0,3.0,2.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
2,55.5,45000.0,3.0,1.0,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,40.5,50000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
4,55.0,55000.0,1.0,1.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,50.5,90000.0,4.0,2.0,0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
4594,37.5,60000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4595,48.0,20000.0,2.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4596,37.5,20000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df_men = df[df["sex"] == 1]
df_women = df[df["sex"] == 2]

In [7]:
def firth_logit(X, y, max_iter=200, tol=1e-7):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    beta = np.zeros(p)
    converged = False

    for _ in range(max_iter):
        eta = np.clip(X @ beta, -30.0, 30.0)
        mu = 1.0 / (1.0 + np.exp(-eta))
        w = mu * (1.0 - mu)
        sqrt_w = np.sqrt(w)

        XW = X * sqrt_w[:, None]
        info = XW.T @ XW
        try:
            info_inv = np.linalg.inv(info)
        except np.linalg.LinAlgError:
            info_inv = np.linalg.pinv(info)

        h = np.einsum("ij,jk,ik->i", XW, info_inv, XW)
        u_star = X.T @ (y - mu + h * (0.5 - mu))
        delta = info_inv @ u_star
        beta = beta + delta

        if np.max(np.abs(delta)) < tol:
            converged = True
            break

    eta = np.clip(X @ beta, -30.0, 30.0)
    mu = 1.0 / (1.0 + np.exp(-eta))
    w = mu * (1.0 - mu)
    info = (X * w[:, None]).T @ X
    try:
        cov = np.linalg.inv(info)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(info)
    se = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    z = np.divide(beta, se, out=np.zeros_like(beta), where=se > 0)
    pval = 2.0 * (1.0 - norm.cdf(np.abs(z)))
    return beta, se, pval, converged


def firth_predict(X, beta):
    eta = np.clip(np.asarray(X, dtype=float) @ beta, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-eta))

In [8]:
def perform_matching(
    data,
    treatment_col,
    outcomes,
    match_covars,
    outcome_covars,
    method="psm",
    caliper_sd=0.2,
):
    all_vars = list(set(match_covars + outcome_covars + [treatment_col] + outcomes))
    data_clean = data.dropna(subset=all_vars).copy().reset_index(drop=True)

    T = data_clean[treatment_col].astype(int).values
    results = {}

    scaler = StandardScaler()
    X_match_scaled = scaler.fit_transform(data_clean[match_covars])
    X_match_scaled_df = pd.DataFrame(
        X_match_scaled, index=data_clean.index, columns=match_covars
    )

    if method == "psm":
        lr = LogisticRegression(C=1e9, max_iter=2000, solver="lbfgs")
        lr.fit(X_match_scaled, T)
        ps = lr.predict_proba(X_match_scaled)[:, 1]

        eps = 1e-6
        ps_clipped = np.clip(ps, eps, 1 - eps)
        logit_ps = np.log(ps_clipped / (1 - ps_clipped))
        caliper = caliper_sd * np.std(logit_ps)

        data_clean["ps"] = ps
        data_clean["logit_ps"] = logit_ps

        ps_treated = ps[T == 1]
        ps_untreated = ps[T == 0]
        lo = max(ps_treated.min(), ps_untreated.min())
        hi = min(ps_treated.max(), ps_untreated.max())
        on_support = (ps >= lo) & (ps <= hi)
        data_clean = data_clean.loc[on_support].reset_index(drop=True)
        T = data_clean[treatment_col].astype(int).values

        treated = data_clean[T == 1]
        untreated = data_clean[T == 0]

        nn = NearestNeighbors(n_neighbors=1).fit(untreated[["logit_ps"]].values)
        dist, idx = nn.kneighbors(treated[["logit_ps"]].values)
        keep = dist.flatten() <= caliper

        treated_kept = treated.iloc[keep]
        controls_kept = untreated.iloc[idx.flatten()[keep]]
        matched_data = pd.concat([treated_kept, controls_kept]).reset_index(drop=True)

    elif method == "mahalanobis":
        treated_scaled = X_match_scaled_df[T == 1]
        untreated_scaled = X_match_scaled_df[T == 0]

        cov = np.cov(X_match_scaled.T)
        vi = np.linalg.pinv(cov)
        nn = NearestNeighbors(
            n_neighbors=1, metric="mahalanobis", metric_params={"VI": vi}
        ).fit(untreated_scaled.values)

        dist, idx = nn.kneighbors(treated_scaled.values)
        keep = dist.flatten() <= np.quantile(dist.flatten(), 0.95)

        treated_idx = treated_scaled.index[keep]
        control_idx = untreated_scaled.index[idx.flatten()[keep]]
        matched_data = pd.concat([
            data_clean.loc[treated_idx],
            data_clean.loc[control_idx],
        ]).reset_index(drop=True)

    treat_idx = None

    for outcome in outcomes:
        y = matched_data[outcome].values
        if np.unique(y).size <= 1 or y.sum() < 5 or (len(y) - y.sum()) < 5:
            results[outcome] = {"pval": np.nan, "effect": np.nan}
            continue

        X_df = matched_data[outcome_covars].copy()
        cont_cols = [c for c in outcome_covars if matched_data[c].nunique() > 5]
        if cont_cols:
            X_df[cont_cols] = StandardScaler().fit_transform(X_df[cont_cols])
        X_df.insert(0, "const", 1.0)
        X_df[treatment_col] = matched_data[treatment_col].values
        X_mat = X_df.values
        if treat_idx is None:
            treat_idx = list(X_df.columns).index(treatment_col)

        beta, se, pval, conv = firth_logit(X_mat, y)
        if not conv or not np.isfinite(pval[treat_idx]):
            results[outcome] = {"pval": np.nan, "effect": np.nan}
            continue

        X_t1 = X_mat.copy()
        X_t1[:, treat_idx] = 1.0
        X_t0 = X_mat.copy()
        X_t0[:, treat_idx] = 0.0
        ate = float((firth_predict(X_t1, beta) - firth_predict(X_t0, beta)).mean())

        results[outcome] = {"pval": float(pval[treat_idx]), "effect": ate}

    return results


treatment = "diploma"

matching_covariates = [
    "age",
    "income",
    "n_child",
    "type_area",
    "invalid",
    "visit_doctor",
    "work",
]

outcome_covariates = matching_covariates

final_tables = {}
for g_name, g_df in [("Men", df_men), ("Women", df_women)]:
    psm_res = perform_matching(
        g_df, treatment, targets, matching_covariates, outcome_covariates, method="psm"
    )
    mah_res = perform_matching(
        g_df,
        treatment,
        targets,
        matching_covariates,
        outcome_covariates,
        method="mahalanobis",
    )

    combined = pd.DataFrame({
        "Disease": targets,
        "PSM Effect": [psm_res.get(t, {}).get("effect", np.nan) for t in targets],
        "PSM P-Value": [psm_res.get(t, {}).get("pval", np.nan) for t in targets],
        "Mahalanobis Effect": [
            mah_res.get(t, {}).get("effect", np.nan) for t in targets
        ],
        "Mahalanobis P-Value": [
            mah_res.get(t, {}).get("pval", np.nan) for t in targets
        ],
    })
    combined["PSM Sig"] = combined["PSM P-Value"] < 0.05
    combined["Mah Sig"] = combined["Mahalanobis P-Value"] < 0.05
    final_tables[g_name] = combined

In [9]:
final_tables["Men"].to_latex("../assets/men.tex")
final_tables["Men"]

,Disease,PSM Effect,PSM P-Value,Mahalanobis Effect,Mahalanobis P-Value,PSM Sig,Mah Sig
0,heart,0.006771,0.613289,0.010431,0.447580,False,False
1,lungs,-0.007612,0.516663,0.011500,0.289365,False,False
2,liver,0.002069,0.846472,0.007135,0.500164,False,False
3,kidneys,-0.005952,0.535805,-0.013959,0.177250,False,False
4,stomach,-0.030666,0.142248,0.005969,0.765822,False,False
5,spine,-0.051948,0.008149,-0.029763,0.121451,True,False
6,diabetes,0.008833,0.420205,0.007022,0.535545,False,False
7,hypertension,0.044188,0.033803,0.061849,0.003107,True,True
8,joints,0.012423,0.485873,0.037476,0.033584,False,True
9,ENT_organs,-0.006653,0.649181,0.031285,0.019815,False,True


In [10]:
final_tables["Women"].to_latex("../assets/women.tex")
final_tables["Women"]

,Disease,PSM Effect,PSM P-Value,Mahalanobis Effect,Mahalanobis P-Value,PSM Sig,Mah Sig
0,heart,-0.000550,0.942303,0.006136,0.409387,False,False
1,lungs,-0.045433,0.000004,-0.043757,0.000016,True,True
2,liver,0.001029,0.879097,0.001087,0.876953,False,False
3,kidneys,-0.027697,0.003204,-0.015143,0.090014,True,False
4,stomach,-0.019860,0.183011,0.005131,0.727734,False,False
5,spine,-0.034387,0.014383,-0.019288,0.170364,True,False
6,diabetes,-0.007126,0.491016,-0.027271,0.015632,False,True
7,hypertension,-0.005844,0.697940,-0.024136,0.114841,False,False
8,joints,-0.042844,0.001622,-0.025883,0.059233,True,False
9,ENT_organs,-0.007646,0.504417,-0.000138,0.990395,False,False


### Анализ результатов

Выше представлена часть из 60 моделей (15 заболеваний × 2 пола × 2 метода).

1. **Propensity Score Matching (PSM)**: Позволяет сбалансировать группы образованных и необразованных по ковариатам (возраст, доход и т.д.) на основе вероятности получения воздействия.
2. **Mahalanobis Distance**: Проводит поиск «близнецов» на основе многомерного расстояния.

Если `P-Value < 0.05`, это означает, что после выравнивания групп выбранный фактор сохраняет статистически значимую связь с конкретным заболеванием у данного пола.